# COSC2753 | Machine Learning

## Task 2: Season Classification

# 1. Introduction

This notebook is the Task 2 starter experiment. **Notebook outputs are intentionally cleared. The existing checkpoint is a prototype used for application testing, not the final investigation. Member 3 must run both controlled candidates, complete every analysis prompt, and replace the mock checkpoint.** Season can be subjective and weakly visible, so evaluation emphasizes macro F1, calibration, article-type confounding, and representative failures rather than accuracy alone.

# 2. Library Imports and Setup

In [ ]:
from copy import deepcopy
from pathlib import Path
import json, sys, time

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from app.server.utils.classifier import FashionClassifier
from app.server.utils.handcrafted import DEFAULT_FEATURE_CONFIG, handcrafted_feature
from evaluation import calibration_table, expected_calibration_error, high_confidence_errors, ordered_estimator_probabilities, saved_model_robustness, select_validation_candidate, subgroup_metrics
from preprocessing import IMAGE_SIZE, NORMALISATION_PATH, SEED, seed_everything, select_torch_device, task_frame
seed_everything(SEED); torch.manual_seed(SEED)
DEVICE = select_torch_device()
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
DEVICE

# 3. Frozen Data and Season EDA

In [ ]:
TARGET = 'season'
train_df = task_frame(TARGET, 'train')
validation_df = task_frame(TARGET, 'validation')
test_df = task_frame(TARGET, 'test')
with NORMALISATION_PATH.open(encoding='utf-8') as handle:
    normalisation = json.load(handle)
labels = sorted(train_df[TARGET].unique())
label_to_index = {label: index for index, label in enumerate(labels)}
def supported_macro_f1(truth, predictions):
    return f1_score(truth, predictions, labels=np.unique(truth), average='macro', zero_division=0)
pd.DataFrame({
    'train': train_df[TARGET].value_counts(),
    'validation': validation_df[TARGET].value_counts(),
    'test': test_df[TARGET].value_counts(),
}).fillna(0).astype(int)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=train_df, x=TARGET, order=train_df[TARGET].value_counts().index, ax=axes[0])
axes[0].set_title('Season imbalance in frozen training data')
pd.crosstab(train_df['articleType'], train_df[TARGET], normalize='index').head(20).plot(kind='bar', stacked=True, ax=axes[1], legend=False)
axes[1].set_title('Season distribution within article types')
plt.tight_layout()

## 3.1 Observations

Discuss Summer dominance, Spring support, blank-label exclusion, and whether garment type may act as a shortcut. Confirm that all validation/test labels exist in training before continuing.

# 4. Majority Baseline

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(train_df), 1)), train_df[TARGET])
dummy_predictions = dummy.predict(np.zeros((len(validation_df), 1)))
dummy_probabilities = ordered_estimator_probabilities(dummy, np.zeros((len(validation_df), 1)), labels)
majority_metrics = {'accuracy': accuracy_score(validation_df[TARGET], dummy_predictions), 'macro_f1': supported_macro_f1(validation_df[TARGET], dummy_predictions)}
majority_metrics

# 5. HOG + HSV Classical Baseline

In [ ]:
def handcrafted_feature_path(path):
    with Image.open(path) as source:
        return handcrafted_feature(source, DEFAULT_FEATURE_CONFIG)

train_features = np.vstack([handcrafted_feature_path(path) for path in train_df.image_path])
validation_features = np.vstack([handcrafted_feature_path(path) for path in validation_df.image_path])
linear_baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs'),
).fit(train_features, train_df[TARGET])
linear_predictions = linear_baseline.predict(validation_features)
linear_probabilities = ordered_estimator_probabilities(linear_baseline, validation_features, labels)
linear_metrics = {'accuracy': accuracy_score(validation_df[TARGET], linear_predictions), 'macro_f1': supported_macro_f1(validation_df[TARGET], linear_predictions)}
linear_metrics

# 6. Image Dataset and Augmentation

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])), transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(8, translate=(0.05, 0.05)), transforms.ColorJitter(0.12, 0.12),
    transforms.ToTensor(), transforms.Normalize(normalisation['mean'], normalisation['std']),
])
evaluation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])), transforms.ToTensor(),
    transforms.Normalize(normalisation['mean'], normalisation['std']),
])

class FashionDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame, self.transform = frame.reset_index(drop=True), transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, label_to_index[row[TARGET]]

train_loader = DataLoader(FashionDataset(train_df, train_transform), 64, shuffle=True, num_workers=0)
validation_loader = DataLoader(FashionDataset(validation_df, evaluation_transform), 128, num_workers=0)
test_loader = DataLoader(FashionDataset(test_df, evaluation_transform), 128, num_workers=0)

# 7. Compact Residual CNN From Scratch

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, input_channels, output_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(output_channels)
        self.conv2 = nn.Conv2d(output_channels, output_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(output_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity() if input_channels == output_channels and stride == 1 else nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 1, stride, bias=False), nn.BatchNorm2d(output_channels),
        )
    def forward(self, inputs):
        residual = self.shortcut(inputs)
        outputs = self.relu(self.bn1(self.conv1(inputs)))
        outputs = self.bn2(self.conv2(outputs))
        return self.relu(outputs + residual)

class CompactCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            ResidualBlock(3, 32), nn.MaxPool2d(2), ResidualBlock(32, 64), nn.MaxPool2d(2),
            ResidualBlock(64, 128), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, inputs): return self.classifier(self.features(inputs))

sum(parameter.numel() for parameter in CompactCNN(len(labels)).parameters())

# 8. Weighted-Loss Training

In [ ]:
counts = train_df[TARGET].value_counts().reindex(labels).values
class_weights = torch.tensor(len(train_df) / (len(labels) * counts), dtype=torch.float32, device=DEVICE)
def train_candidate(loss_mode, epochs=40, patience_limit=7):
    if loss_mode not in {'ordinary', 'weighted'}:
        raise ValueError('loss_mode must be ordinary or weighted')
    seed_everything(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    generator = torch.Generator().manual_seed(SEED)
    candidate_loader = DataLoader(FashionDataset(train_df, train_transform), 64, shuffle=True, num_workers=0, generator=generator)
    candidate = CompactCNN(len(labels)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=None if loss_mode == 'ordinary' else class_weights)
    optimizer = torch.optim.AdamW(candidate.parameters(), lr=1e-3, weight_decay=1e-4)

    def run_epoch(loader, training=False):
        candidate.train(training); losses, truth, predictions = [], [], []
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(training):
                logits = candidate(images); loss = criterion(logits, targets)
                if training: loss.backward(); optimizer.step()
            losses.append(loss.item() * len(targets)); truth.extend(targets.cpu().tolist()); predictions.extend(logits.argmax(1).cpu().tolist())
        return {'loss': sum(losses) / len(loader.dataset), 'accuracy': accuracy_score(truth, predictions), 'macro_f1': supported_macro_f1(truth, predictions)}

    history, best_state, best_f1, patience = [], None, -1.0, 0
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        train_metrics, validation_metrics = run_epoch(candidate_loader, True), run_epoch(validation_loader)
        history.append({'epoch': epoch, 'loss_mode': loss_mode, **{f'train_{key}': value for key, value in train_metrics.items()}, **{f'validation_{key}': value for key, value in validation_metrics.items()}})
        print(loss_mode, epoch, validation_metrics, f'{time.perf_counter() - started:.1f}s')
        if validation_metrics['macro_f1'] > best_f1:
            best_f1, best_state, patience = validation_metrics['macro_f1'], deepcopy(candidate.state_dict()), 0
        else:
            patience += 1
            if patience >= patience_limit: break
    candidate.load_state_dict(best_state)
    return candidate, pd.DataFrame(history), best_f1

In [ ]:
candidate_runs = {mode: train_candidate(mode) for mode in ('ordinary', 'weighted')}
validation_truth = np.asarray([label_to_index[label] for label in validation_df[TARGET]])
@torch.inference_mode()
def validation_probabilities(model):
    model.eval(); values = []
    for images, _ in validation_loader:
        values.append(model(images.to(DEVICE)).softmax(1).cpu().numpy())
    return np.vstack(values)
comparison_rows = [
    {'method': 'majority', 'model_type': 'reference_only', 'validation_accuracy': majority_metrics['accuracy'], 'validation_macro_f1': majority_metrics['macro_f1'], 'validation_ece': expected_calibration_error(validation_truth, dummy_probabilities), 'complexity_parameters': 1, 'epochs_run': 0, 'eligible_for_selection': False},
    {'method': 'hog_hsv_logistic_regression', 'model_type': 'hog_hsv_logistic_regression', 'validation_accuracy': linear_metrics['accuracy'], 'validation_macro_f1': linear_metrics['macro_f1'], 'validation_ece': expected_calibration_error(validation_truth, linear_probabilities), 'complexity_parameters': linear_baseline[-1].coef_.size + linear_baseline[-1].intercept_.size, 'epochs_run': 0, 'eligible_for_selection': True},
]
for loss_mode, result in candidate_runs.items():
    probabilities = validation_probabilities(result[0])
    comparison_rows.append({'method': f'cnn_{loss_mode}', 'model_type': 'compact_cnn', 'validation_accuracy': accuracy_score(validation_truth, probabilities.argmax(1)), 'validation_macro_f1': result[2], 'validation_ece': expected_calibration_error(validation_truth, probabilities), 'complexity_parameters': sum(parameter.numel() for parameter in result[0].parameters()), 'epochs_run': len(result[1]), 'eligible_for_selection': True})
comparison = pd.DataFrame(comparison_rows).set_index('method')
display(comparison)
SELECTED_METHOD = select_validation_candidate(comparison.loc[comparison.eligible_for_selection])
SELECTED_MODEL_TYPE = comparison.loc[SELECTED_METHOD, 'model_type']
best_f1 = float(comparison.loc[SELECTED_METHOD, 'validation_macro_f1'])
if SELECTED_MODEL_TYPE == 'compact_cnn':
    SELECTED_LOSS_MODE = SELECTED_METHOD.removeprefix('cnn_')
    model, history, _ = candidate_runs[SELECTED_LOSS_MODE]
    history.plot(x='epoch', y=['train_macro_f1', 'validation_macro_f1'], figsize=(8, 4), title='Selected season learning curve'); plt.show()
else:
    SELECTED_LOSS_MODE = None
    model = linear_baseline
    history = pd.DataFrame([{'method': SELECTED_METHOD, **comparison.loc[SELECTED_METHOD].to_dict()}])
print(f'Selected {SELECTED_METHOD!r} before opening the internal-test section.')

## 7.1 Controlled comparison

The preceding cell runs ordinary and weighted cross-entropy from identical seeds and fresh model/data-loader state. Confirm the selected candidate from validation macro F1 and calibration evidence before opening the internal-test section.

# 9. Final Evaluation

In [ ]:
@torch.inference_mode()
def predict(loader):
    model.eval(); probabilities, truth = [], []
    for images, targets in loader:
        probabilities.append(model(images.to(DEVICE)).softmax(1).cpu().numpy()); truth.extend(targets.tolist())
    return np.vstack(probabilities), np.asarray(truth)

if SELECTED_MODEL_TYPE == 'compact_cnn':
    test_probabilities, test_truth = predict(test_loader)
else:
    test_features = np.vstack([handcrafted_feature_path(path) for path in test_df.image_path])
    test_probabilities = ordered_estimator_probabilities(model, test_features, labels)
    test_truth = np.asarray([label_to_index[label] for label in test_df[TARGET]])
test_predictions = test_probabilities.argmax(1)
test_metrics = {'accuracy': accuracy_score(test_truth, test_predictions), 'macro_f1': supported_macro_f1(test_truth, test_predictions)}
test_metrics

In [ ]:
print(classification_report(
    test_truth, test_predictions, labels=np.arange(len(labels)), target_names=labels, zero_division=0,
))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    test_truth, test_predictions, labels=np.arange(len(labels)), display_labels=labels,
    normalize='true', ax=ax, cmap='Blues',
)
plt.show()

## 8.1 Calibration and failure analysis

In [ ]:
display(calibration_table(test_truth, test_probabilities))
display(high_confidence_errors(test_df, TARGET, labels, test_truth, test_predictions, test_probabilities))
truth_labels = [labels[index] for index in test_truth]
prediction_labels = [labels[index] for index in test_predictions]
display(subgroup_metrics(test_df, truth_labels, prediction_labels, group_column='articleType', minimum_support=30))

Analyse high-confidence mistakes by article type and explain whether the model learned garment cues or seasonal priors. Include mild brightness/blur robustness, inference latency, checkpoint size, and the subjective nature of season labels.

# 10. Save the Selected Model

In [ ]:
checkpoint_path = ROOT / 'models' / 'season_model.pt'
validation_metrics = {key: float(comparison.loc[SELECTED_METHOD, key]) for key in ('validation_accuracy', 'validation_macro_f1', 'validation_ece')}
if SELECTED_MODEL_TYPE == 'compact_cnn':
    checkpoint = {
        'model_type': 'compact_cnn', 'target': TARGET, 'labels': labels,
        'state_dict': model.cpu().state_dict(), 'mean': normalisation['mean'], 'std': normalisation['std'],
        'image_size': list(IMAGE_SIZE), 'dropout': 0.2, 'loss_mode': SELECTED_LOSS_MODE,
    }
else:
    checkpoint = {
        'model_type': 'hog_hsv_logistic_regression', 'target': TARGET, 'labels': labels,
        'estimator': model, 'feature_config': DEFAULT_FEATURE_CONFIG,
    }
checkpoint.update({'selection_metric': 'validation_macro_f1', 'best_validation_macro_f1': best_f1, 'validation_metrics': validation_metrics, 'test_metrics': test_metrics, 'seed': SEED})
torch.save(checkpoint, checkpoint_path)
cpu_predictor = FashionClassifier(checkpoint_path, device='cpu')
robustness = saved_model_robustness(cpu_predictor, test_df, TARGET, SEED)
display(robustness)
print(f'Checkpoint size: {checkpoint_path.stat().st_size / 1024**2:.2f} MiB')
history.to_csv(ROOT / 'models' / 'season_history.csv', index=False)
comparison.to_csv(ROOT / 'models' / 'season_comparison.csv')
checkpoint_path

# 11. Final Prediction

Run `python scripts/task2_season_classification.py path/to/image.jpg` from the repository root to verify the saved artifact on CPU.

# 12. Ultimate Judgement and Conclusion

Replace this prompt with the real comparison among majority, HOG+HSV, ordinary-loss residual CNN, and weighted residual CNN; include class-specific findings, calibration, confounding evidence, limitations, and responsible interpretation of image-based season prediction.